# MCP（Model Context Protocol）

- MCPの概念とツール定義との比較
- MCPクライアントの通信フロー
- シンプルなMCPサーバーの実装と接続

## 1. セットアップ

In [165]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

print("セットアップ完了")

セットアップ完了


## 2. MCPとは

**MCPとは：** ツール定義と実行の負担を、自分のサーバーからMCPサーバーに移行する通信レイヤー。

**MCPなしの場合：**
- GitHubのAPIを使いたければ、ツールスキーマと関数を全部自分で書く
- 機能が増えるたびに実装・テスト・保守が必要

**MCPありの場合：**
- GitHubのMCPサーバーに接続するだけ
- ツールはMCPサーバー側に既に定義・実装済み

```
【MCPなし】
あなたのサーバー
  ├── get_repositories() ← 自分で実装
  ├── list_pull_requests() ← 自分で実装
  └── create_issue() ← 自分で実装

【MCPあり】
あなたのサーバー
  └── GitHub MCPサーバーに接続するだけ
        ├── get_repositories (実装済み)
        ├── list_pull_requests (実装済み)
        └── create_issue (実装済み)
```

**MCPサーバーは誰が作る？**
- サービスプロバイダー自身（例：AWS、GitHub が公式に提供）
- コミュニティ（オープンソース）
- 自分自身（社内ツールをMCPサーバーとして公開）

In [166]:
# 【比較】MCPなし：ツールを全部自分で定義・実装する
def get_repositories(username: str) -> str:
    """GitHubのリポジトリ一覧を取得（ダミー実装）"""
    return f"{username} のリポジトリ: repo-a, repo-b, repo-c"

def list_pull_requests(repo: str) -> str:
    """プルリクエスト一覧を取得（ダミー実装）"""
    return f"{repo} のPR: #1 Fix bug, #2 Add feature"

# ツールスキーマも自分で定義する必要がある
tools_without_mcp = [
    {
        "name": "get_repositories",
        "description": "GitHubユーザーのリポジトリ一覧を取得する",
        "input_schema": {
            "type": "object",
            "properties": {"username": {"type": "string"}},
            "required": ["username"],
        },
    },
    {
        "name": "list_pull_requests",
        "description": "リポジトリのPR一覧を取得する",
        "input_schema": {
            "type": "object",
            "properties": {"repo": {"type": "string"}},
            "required": ["repo"],
        },
    },
]

print("MCPなし：スキーマ定義と関数実装を両方自分で書く必要がある")
print(f"定義したツール数: {len(tools_without_mcp)}")
print("→ GitHub の全機能をカバーしようとすると数十〜数百のツールが必要になる")

MCPなし：スキーマ定義と関数実装を両方自分で書く必要がある
定義したツール数: 2
→ GitHub の全機能をカバーしようとすると数十〜数百のツールが必要になる


In [167]:
# 【比較】MCPあり：MCPサーバーのURLを指定するだけ
# ツール定義も関数実装も不要 → MCPサーバー側に既にある

# ※ 実際に動かすには接続先のMCPサーバーが必要（以下は構造の説明）
mcp_tool = {
    "type": "mcp",                          # ツールタイプを "mcp" に指定
    "server_label": "github",               # 識別ラベル（任意の名前）
    "server_url": "https://mcp.example.com/github",  # MCPサーバーのURL
    # 必要に応じて認証ヘッダーを追加
    # "headers": {"Authorization": f"Bearer {github_token}"},
    # 使用を許可するツールを絞る場合（省略すると全ツールが使える）
    # "allowed_tools": ["get_repositories", "list_pull_requests"],
}

# API呼び出しの構造（実行にはMCPサーバーへの接続が必要）
params = {
    "model": model,
    "max_tokens": 1024,
    "tools": [mcp_tool],                    # スキーマ定義なし・接続先URLだけ
    "messages": [{"role": "user", "content": "オープンなPRを全て教えてください"}],
    "betas": ["mcp-client-2025-04-04"],     # MCPを使う場合のbetaフラグ
}

print("MCPあり：tools に接続先URLを書くだけ")
print("  - ツールスキーマの定義: 不要（MCPサーバー側が持っている）")
print("  - 関数の実装: 不要（MCPサーバー側が実行する）")
print("  - 保守: 不要（MCPサーバー側がアップデートしてくれる）")

MCPあり：tools に接続先URLを書くだけ
  - ツールスキーマの定義: 不要（MCPサーバー側が持っている）
  - 関数の実装: 不要（MCPサーバー側が実行する）
  - 保守: 不要（MCPサーバー側がアップデートしてくれる）


## 3. MCPクライアントの通信フロー

MCPクライアントはあなたのサーバーとMCPサーバーの間の「通信ブリッジ」として動作します。

**トランスポート方式（通信手段）：**
- `stdio`（標準入出力）：同一マシン上で最もシンプルな構成
- `HTTP` / `WebSocket`：ネットワーク越しにMCPサーバーに接続

**主なメッセージタイプ：**

| メッセージ | 方向 | 内容 |
|---|---|---|
| `ListToolsRequest` | クライアント → サーバー | 「どんなツールがある？」 |
| `ListToolsResult` | サーバー → クライアント | ツール一覧を返す |
| `CallToolRequest` | クライアント → サーバー | 「このツールをこの引数で実行して」 |
| `CallToolResult` | サーバー → クライアント | 実行結果を返す |

**完全な通信フロー（例：「自分のリポジトリは何ですか？」）：**
```
ユーザー
  → あなたのサーバー
    → MCPクライアント --[ListToolsRequest]--> MCPサーバー
    ← MCPクライアント <--[ListToolsResult]--- MCPサーバー
  → Claude（ツール一覧 + ユーザーの質問を送信）
  ← Claude（ツール使用リクエストを返す）
    → MCPクライアント --[CallToolRequest]--> MCPサーバー --> GitHub API
    ← MCPクライアント <--[CallToolResult]--- MCPサーバー <-- GitHub API
  → Claude（ツール実行結果を送信）
  ← Claude（最終回答を返す）
ユーザー ← 回答
```

In [168]:
%pip install mcp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 16.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [mcp]5/9 [cryptography]
Note: you may need to restart the kernel to use updated packages.


In [169]:
# シンプルなMCPサーバーをファイルとして作成する
# stdio トランスポートで動作し、天気情報ツールを提供するサーバー

mcp_server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("weather-server")

@mcp.tool()
def get_weather(city: str) -> str:
    """指定した都市の現在の天気を返す"""
    # ダミーデータ（実際はAPIを呼ぶ）
    data = {
        "東京": "晴れ、気温22℃",
        "大阪": "曇り、気温20℃",
        "札幌": "雪、気温-2℃",
    }
    return data.get(city, f"{city} のデータは見つかりませんでした")

@mcp.tool()
def get_forecast(city: str, days: int = 3) -> str:
    """指定した都市の天気予報を返す"""
    return f"{city} の{days}日間予報: 晴れ→曇り→雨"

if __name__ == "__main__":
    mcp.run()  # stdio トランスポートで起動
'''

with open("./mcp_server.py", "w") as f:
    f.write(mcp_server_code)

print("MCPサーバーファイルを作成: ./mcp_server.py")
print()
print("サーバーが提供するツール：")
print("  - get_weather(city)       : 現在の天気")
print("  - get_forecast(city, days): 天気予報")

MCPサーバーファイルを作成: ./mcp_server.py

サーバーが提供するツール：
  - get_weather(city)       : 現在の天気
  - get_forecast(city, days): 天気予報


In [172]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def run_mcp_flow(user_question: str):
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["mcp_server.py"],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # ① ListToolsRequest：MCPサーバーからツール一覧を取得
            print("=== ① ListToolsRequest ===")
            tools_result = await session.list_tools()
            print(f"取得したツール数: {len(tools_result.tools)}")
            for t in tools_result.tools:
                print(f"  - {t.name}: {t.description}")

            tools_for_claude = [
                {"name": t.name, "description": t.description, "input_schema": t.inputSchema}
                for t in tools_result.tools
            ]

            print(f"\n=== ② Claude に質問: '{user_question}' ===")
            messages = [{"role": "user", "content": user_question}]
            response = client.messages.create(
                model=model, max_tokens=512,
                tools=tools_for_claude, messages=messages,
            )

            # ③ Claude がツール使用を要求した場合：1レスポンスに複数のtool_useがある場合もまとめて処理
            while response.stop_reason == "tool_use":
                tool_uses = [b for b in response.content if b.type == "tool_use"]

                # 全 tool_use を実行して tool_result をまとめる
                tool_results = []
                for tool_use in tool_uses:
                    print(f"\n=== ③ CallToolRequest: {tool_use.name}({tool_use.input}) ===")
                    tool_result = await session.call_tool(tool_use.name, tool_use.input)
                    result_text = tool_result.content[0].text
                    print(f"CallToolResult: {result_text}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": tool_use.id,
                        "content": result_text,
                    })

                # 全 tool_result を1つのメッセージにまとめて返す
                messages += [
                    {"role": "assistant", "content": response.content},
                    {"role": "user", "content": tool_results},
                ]
                response = client.messages.create(
                    model=model, max_tokens=512,
                    tools=tools_for_claude, messages=messages,
                )

            print(f"\n=== ④ 最終回答 ===")
            print(response.content[0].text)


await run_mcp_flow("東京と大阪の天気を教えてください。")

=== ① ListToolsRequest ===
取得したツール数: 2
  - get_weather: 指定した都市の現在の天気を返す
  - get_forecast: 指定した都市の天気予報を返す

=== ② Claude に質問: '東京と大阪の天気を教えてください。' ===

=== ③ CallToolRequest: get_weather({'city': '東京'}) ===
CallToolResult: 晴れ、気温22℃

=== ③ CallToolRequest: get_weather({'city': '大阪'}) ===
CallToolResult: 曇り、気温20℃

=== ④ 最終回答 ===
東京と大阪の現在の天気は以下の通りです:

**東京**: 晴れ、気温22℃
**大阪**: 曇り、気温20℃


## 4. MCPリソース

**リソースとツールの違い：**
- **ツール**：アクションを実行する（副作用あり）→ `POST` 的
- **リソース**：データを取得する（副作用なし）→ `GET` 的

**リソースの種類：**
- **直接リソース**：固定URI、例：`docs://documents`
- **テンプレート化されたリソース**：パラメータ付きURI、例：`docs://documents/{doc_id}`

**ユースケース例：** `@ドキュメント名` のオートコンプリート機能
1. `docs://documents` → 利用可能なドキュメント一覧（オートコンプリート用）
2. `docs://documents/{doc_id}` → 特定ドキュメントの内容取得

In [173]:
# mcp_server.py にリソース定義を追加する
# ツール（天気）に加えて、ドキュメント参照リソースを追加

mcp_server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("weather-server")

# ========== ツール ==========

@mcp.tool()
def get_weather(city: str) -> str:
    """指定した都市の現在の天気を返す"""
    data = {
        "東京": "晴れ、気温22℃",
        "大阪": "曇り、気温20℃",
        "札幌": "雪、気温-2℃",
    }
    return data.get(city, f"{city} のデータは見つかりませんでした")

@mcp.tool()
def get_forecast(city: str, days: int = 3) -> str:
    """指定した都市の天気予報を返す"""
    return f"{city} の{days}日間予報: 晴れ→曇り→雨"

# ========== リソース ==========

# ドキュメントのダミーデータ
docs = {
    "api-guide":    "# API ガイド\\nこのドキュメントはAPIの使い方を説明します。\\n\\n## 認証\\nBearerトークンを使用してください。",
    "quickstart":   "# クイックスタート\\n1. パッケージをインストール\\n2. APIキーを設定\\n3. コードを実行",
    "faq":          "# よくある質問\\nQ: レート制限は？\\nA: 1分間に60リクエストまでです。",
}

# 直接リソース：ドキュメント一覧（固定URI）
@mcp.resource(
    "docs://documents",
    mime_type="application/json"
)
def list_docs() -> list[str]:
    """利用可能なドキュメントの一覧を返す"""
    return list(docs.keys())

# テンプレート化されたリソース：特定ドキュメントの取得
@mcp.resource(
    "docs://documents/{doc_id}",
    mime_type="text/plain"
)
def fetch_doc(doc_id: str) -> str:
    """指定したIDのドキュメント内容を返す"""
    if doc_id not in docs:
        raise ValueError(f"ドキュメント '{doc_id}' が見つかりません")
    return docs[doc_id]

if __name__ == "__main__":
    mcp.run()
'''

with open("./mcp_server.py", "w") as f:
    f.write(mcp_server_code)

print("mcp_server.py を更新しました（リソース追加）")
print()
print("追加したリソース：")
print("  - docs://documents           : ドキュメント一覧（直接リソース）")
print("  - docs://documents/{doc_id}  : ドキュメント内容（テンプレート化されたリソース）")

mcp_server.py を更新しました（リソース追加）

追加したリソース：
  - docs://documents           : ドキュメント一覧（直接リソース）
  - docs://documents/{doc_id}  : ドキュメント内容（テンプレート化されたリソース）


In [174]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def run_resource_flow():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["mcp_server.py"],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # ① list_resources：直接リソースの一覧を取得
            print("=== ① list_resources（直接リソース） ===")
            resources_result = await session.list_resources()
            for r in resources_result.resources:
                print(f"  URI: {r.uri}  mimeType: {r.mimeType}")

            # ② list_resource_templates：テンプレート化されたリソースの一覧を取得
            print("\n=== ② list_resource_templates（テンプレート化されたリソース） ===")
            templates_result = await session.list_resource_templates()
            for t in templates_result.resourceTemplates:
                print(f"  URI テンプレート: {t.uriTemplate}")

            # ③ read_resource：直接リソースを読み取る（ドキュメント一覧）
            print("\n=== ③ read_resource: docs://documents ===")
            list_result = await session.read_resource("docs://documents")
            print(f"  内容: {list_result.contents[0].text}")

            # ④ read_resource：テンプレートリソースを読み取る（特定ドキュメント）
            print("\n=== ④ read_resource: docs://documents/quickstart ===")
            doc_result = await session.read_resource("docs://documents/quickstart")
            print(f"  内容:\n{doc_result.contents[0].text}")

            # ⑤ 存在しないドキュメントを取得しようとするケース
            print("\n=== ⑤ 存在しないドキュメント: docs://documents/unknown ===")
            try:
                await session.read_resource("docs://documents/unknown")
            except Exception as e:
                print(f"  エラー: {e}")

await run_resource_flow()

=== ① list_resources（直接リソース） ===
  URI: docs://documents  mimeType: application/json

=== ② list_resource_templates（テンプレート化されたリソース） ===
  URI テンプレート: docs://documents/{doc_id}

=== ③ read_resource: docs://documents ===
  内容: [
  "api-guide",
  "quickstart",
  "faq"
]

=== ④ read_resource: docs://documents/quickstart ===
  内容:
# クイックスタート
1. パッケージをインストール
2. APIキーを設定
3. コードを実行

=== ⑤ 存在しないドキュメント: docs://documents/unknown ===
  エラー: Error creating resource from template: Error creating resource from template: ドキュメント 'unknown' が見つかりません


## 5. リソースをプロンプトに埋め込む

**ツール呼び出しとの比較：**

| 方式 | フロー | ラウンドトリップ |
|---|---|---|
| ツール経由 | Claude → ツール呼び出し → 結果取得 → 回答 | 2回 |
| リソース直接埋め込み | リソース取得 → プロンプトに挿入 → 回答 | 1回 |

リソースの内容を事前にプロンプトへ注入することで、Claudeがツール呼び出しをせずに直接回答できる。

**`@ドキュメント名` メンション機能のフロー：**
```
ユーザーが "@quickstart の手順を要約して" と入力
  → docs://documents からドキュメント一覧を取得（オートコンプリート用）
  → docs://documents/quickstart の内容を取得
  → プロンプトに内容を挿入して Claude に送信
  → Claude がツール呼び出しなしで直接回答
```

In [175]:
import json
import sys
import re
from pydantic import AnyUrl
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client

class DocsMCPClient:
    """ドキュメントリソースを扱うMCPクライアント"""

    def __init__(self, session: ClientSession):
        self._session = session

    async def list_docs(self) -> list[str]:
        """利用可能なドキュメント一覧を返す（オートコンプリート用）"""
        result = await self._session.read_resource(AnyUrl("docs://documents"))
        resource = result.contents[0]
        if isinstance(resource, types.TextResourceContents):
            if resource.mimeType == "application/json":
                return json.loads(resource.text)
        return []

    async def read_resource(self, uri: str) -> str | dict:
        """URIを指定してリソースを取得する。mimeTypeに応じて自動パース"""
        result = await self._session.read_resource(AnyUrl(uri))
        resource = result.contents[0]
        if isinstance(resource, types.TextResourceContents):
            if resource.mimeType == "application/json":
                return json.loads(resource.text)
            return resource.text
        return ""

    async def fetch_mentioned_doc(self, user_message: str) -> tuple[str | None, str | None]:
        """
        メッセージ中の @doc_id を検出してドキュメント内容を返す。
        戻り値: (doc_id, content) または (None, None)
        """
        match = re.search(r"@(\S+)", user_message)
        if not match:
            return None, None

        doc_id = match.group(1)
        available = await self.list_docs()
        if doc_id not in available:
            return doc_id, None

        content = await self.read_resource(f"docs://documents/{doc_id}")
        return doc_id, content

print("DocsMCPClient 定義完了")

DocsMCPClient 定義完了


In [176]:
async def chat_with_doc_mention(user_message: str):
    """
    @doc_id メンションを含むメッセージを処理する。
    ドキュメント内容をプロンプトに直接埋め込んでClaudeに送信する。
    """
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["mcp_server.py"],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            docs_client = DocsMCPClient(session)

            # オートコンプリート用：利用可能なドキュメント一覧を取得
            available_docs = await docs_client.list_docs()
            print(f"利用可能なドキュメント: {available_docs}")
            print(f"\nユーザー入力: '{user_message}'")

            # @メンションを検出してドキュメント内容を取得
            doc_id, doc_content = await docs_client.fetch_mentioned_doc(user_message)

            if doc_id and doc_content:
                # ドキュメント内容をプロンプトに直接埋め込む（ツール呼び出し不要）
                prompt = f"""以下のドキュメント(@{doc_id})の内容を参考に質問に答えてください。

--- ドキュメント: {doc_id} ---
{doc_content}
---

質問: {user_message.replace(f"@{doc_id}", "").strip()}"""
                print(f"\n[プロンプトにドキュメント内容を埋め込みました]")
            elif doc_id:
                prompt = f"ドキュメント '@{doc_id}' は見つかりませんでした。利用可能なドキュメント: {available_docs}"
                print(f"\n[ドキュメント '{doc_id}' が見つかりません]")
            else:
                prompt = user_message
                print("\n[@メンションなし：そのままClaudeに送信]")

            # Claudeに送信（ツール定義なし）
            response = client.messages.create(
                model=model,
                max_tokens=512,
                messages=[{"role": "user", "content": prompt}],
            )

            print(f"\n=== Claudeの回答（ツール呼び出しなし） ===")
            print(response.content[0].text)
            print(f"\nstop_reason: {response.stop_reason}")  # tool_use にならないことを確認


await chat_with_doc_mention("@quickstart の手順を箇条書きで教えてください")

利用可能なドキュメント: ['api-guide', 'quickstart', 'faq']

ユーザー入力: '@quickstart の手順を箇条書きで教えてください'

[プロンプトにドキュメント内容を埋め込みました]

=== Claudeの回答（ツール呼び出しなし） ===
クイックスタートの手順は以下の通りです：

1. パッケージをインストール
2. APIキーを設定
3. コードを実行

stop_reason: end_turn


## 6. MCPプロンプト

**プロンプトとは：** MCPサーバー側であらかじめ定義した、高品質なプロンプトテンプレート。

**なぜサーバー側で定義するのか：**
- クライアントがゼロから作るより、サーバー開発者が綿密にテストしたプロンプトの方が品質が高い
- 各クライアントが同じ高品質なプロンプトを再利用できる
- プロンプトの改善をサーバー側だけで行える（クライアントの変更不要）

**仕組み：**
1. サーバーが `@mcp.prompt()` でプロンプトを定義する
2. クライアントが `list_prompts` で一覧を取得する
3. クライアントが `get_prompt(name, arguments)` でメッセージリストを取得する
4. そのメッセージをそのままClaudeに送信する

In [179]:
# mcp_server.py にプロンプト定義を追加する

mcp_server_code = '''
from mcp.server.fastmcp import FastMCP
from pydantic import Field

mcp = FastMCP("weather-server")

# ========== ツール ==========

@mcp.tool()
def get_weather(city: str) -> str:
    """指定した都市の現在の天気を返す"""
    data = {
        "東京": "晴れ、気温22℃",
        "大阪": "曇り、気温20℃",
        "札幌": "雪、気温-2℃",
    }
    return data.get(city, f"{city} のデータは見つかりませんでした")

@mcp.tool()
def get_forecast(city: str, days: int = 3) -> str:
    """指定した都市の天気予報を返す"""
    return f"{city} の{days}日間予報: 晴れ→曇り→雨"

# ========== リソース ==========

docs = {
    "api-guide":    "# API ガイド\\nこのドキュメントはAPIの使い方を説明します。\\n\\n## 認証\\nBearerトークンを使用してください。",
    "quickstart":   "# クイックスタート\\n1. パッケージをインストール\\n2. APIキーを設定\\n3. コードを実行",
    "faq":          "# よくある質問\\nQ: レート制限は？\\nA: 1分間に60リクエストまでです。",
}

@mcp.resource("docs://documents", mime_type="application/json")
def list_docs() -> list[str]:
    """利用可能なドキュメントの一覧を返す"""
    return list(docs.keys())

@mcp.resource("docs://documents/{doc_id}", mime_type="text/plain")
def fetch_doc(doc_id: str) -> str:
    """指定したIDのドキュメント内容を返す"""
    if doc_id not in docs:
        raise ValueError(f"ドキュメント \'{doc_id}\' が見つかりません")
    return docs[doc_id]

# ========== プロンプト ==========

@mcp.prompt(
    name="summarize",
    description="ドキュメントの内容を簡潔に要約する。"
)
def prompt_summarize(
    doc_id: str = Field(description="要約するドキュメントのID")
) -> str:
    # FastMCP は文字列を返すと自動的に UserMessage にラップする
    if doc_id not in docs:
        raise ValueError(f"ドキュメント \'{doc_id}\' が見つかりません")
    return f"""以下のドキュメントを要約してください。

要約のルール:
- 3行以内にまとめる
- 箇条書きを使う
- 専門用語はそのまま残す

--- ドキュメント: {doc_id} ---
{docs[doc_id]}
---"""

@mcp.prompt(
    name="weather-report",
    description="指定した都市の天気を丁寧なレポート形式で回答する。"
)
def prompt_weather_report(
    city: str = Field(description="天気を調べる都市名")
) -> str:
    return f"""{city}の天気を調べて、以下の形式でレポートしてください。

【天気レポートのフォーマット】
- 都市名を見出しにする
- 現在の天気と気温を最初に記載する
- 3日間の天気予報も含める
- 最後に一言アドバイス（服装・傘など）を添える"""

if __name__ == "__main__":
    mcp.run()
'''

with open("./mcp_server.py", "w") as f:
    f.write(mcp_server_code)

print("mcp_server.py を更新しました（プロンプト追加）")
print()
print("追加したプロンプト：")
print("  - summarize(doc_id)      : ドキュメントを3行で要約")
print("  - weather-report(city)   : 天気をレポート形式で回答")

mcp_server.py を更新しました（プロンプト追加）

追加したプロンプト：
  - summarize(doc_id)      : ドキュメントを3行で要約
  - weather-report(city)   : 天気をレポート形式で回答


In [180]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def run_prompt_flow():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["mcp_server.py"],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # ① list_prompts：利用可能なプロンプト一覧を取得
            print("=== ① list_prompts ===")
            prompts_result = await session.list_prompts()
            for p in prompts_result.prompts:
                args = [f"{a.name}({'必須' if a.required else '任意'})" for a in (p.arguments or [])]
                print(f"  - {p.name}: {p.description}  引数: {args}")

            # ② get_prompt：summarize プロンプトを取得してClaudeに送信
            print("\n=== ② get_prompt: summarize(doc_id=faq) ===")
            prompt_result = await session.get_prompt(
                "summarize",
                arguments={"doc_id": "faq"}
            )
            # サーバーが返したメッセージをそのままClaudeに送信
            messages = [
                {"role": m.role, "content": m.content.text}
                for m in prompt_result.messages
            ]
            print(f"サーバーから受け取ったメッセージ:\n{messages[0]['content'][:80]}...")

            response = client.messages.create(
                model=model,
                max_tokens=512,
                messages=messages,
            )
            print(f"\nClaudeの回答:\n{response.content[0].text}")

            # ③ get_prompt：weather-report プロンプトを取得してClaudeに送信
            print("\n=== ③ get_prompt: weather-report(city=東京) ===")
            prompt_result2 = await session.get_prompt(
                "weather-report",
                arguments={"city": "東京"}
            )
            messages2 = [
                {"role": m.role, "content": m.content.text}
                for m in prompt_result2.messages
            ]

            # weather-report はツールが必要なのでツール定義も渡す
            tools_result = await session.list_tools()
            tools_for_claude = [
                {"name": t.name, "description": t.description, "input_schema": t.inputSchema}
                for t in tools_result.tools
            ]

            response2 = client.messages.create(
                model=model,
                max_tokens=512,
                tools=tools_for_claude,
                messages=messages2,
            )

            # ツール呼び出しが発生した場合は処理する
            while response2.stop_reason == "tool_use":
                tool_uses = [b for b in response2.content if b.type == "tool_use"]
                tool_results = []
                for tool_use in tool_uses:
                    print(f"  ツール呼び出し: {tool_use.name}({tool_use.input})")
                    tool_result = await session.call_tool(tool_use.name, tool_use.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": tool_use.id,
                        "content": tool_result.content[0].text,
                    })
                messages2 += [
                    {"role": "assistant", "content": response2.content},
                    {"role": "user", "content": tool_results},
                ]
                response2 = client.messages.create(
                    model=model, max_tokens=512,
                    tools=tools_for_claude, messages=messages2,
                )

            print(f"\nClaudeの回答:\n{response2.content[0].text}")

await run_prompt_flow()

=== ① list_prompts ===
  - summarize: ドキュメントの内容を簡潔に要約する。  引数: ['doc_id(必須)']
  - weather-report: 指定した都市の天気を丁寧なレポート形式で回答する。  引数: ['city(必須)']

=== ② get_prompt: summarize(doc_id=faq) ===
サーバーから受け取ったメッセージ:
以下のドキュメントを要約してください。

要約のルール:
- 3行以内にまとめる
- 箇条書きを使う
- 専門用語はそのまま残す

--- ドキュメント: fa...

Claudeの回答:
- レート制限は1分間に60リクエストまで
- よくある質問(FAQ)に関する内容
- API利用時の制限事項を説明

=== ③ get_prompt: weather-report(city=東京) ===
  ツール呼び出し: get_weather({'city': '東京'})
  ツール呼び出し: get_forecast({'city': '東京', 'days': 3})

Claudeの回答:
# 東京

**現在の天気**  
晴れ、気温22℃

**3日間の天気予報**
- 1日目: 晴れ
- 2日目: 曇り
- 3日目: 雨

**アドバイス**  
現在は過ごしやすい気温ですが、3日目には雨の予報となっています。外出の際は折りたたみ傘を携帯されることをおすすめします。また、気温は22℃と快適ですので、軽めの長袖や羽織るものがあると良いでしょう。


## 7. プロンプトを扱うクライアントクラス

前のSTEPでは `run_prompt_flow` に処理をまとめて書いたが、実際のアプリケーションでは再利用可能なクライアントクラスとして切り出す。

**CLIでの操作イメージ：**
```
> /                         ← スラッシュを入力
利用可能なプロンプト:
  /summarize   - ドキュメントの内容を簡潔に要約する。
  /weather-report - 指定した都市の天気を丁寧なレポート形式で回答する。

> /summarize               ← プロンプトを選択
引数 doc_id を入力: faq    ← 引数を入力
（Claudeへ送信 → 回答）
```

In [181]:
from mcp import ClientSession, types

class PromptMCPClient:
    """プロンプト操作に特化したMCPクライアント"""

    def __init__(self, session: ClientSession):
        self._session = session

    async def list_prompts(self) -> list[types.Prompt]:
        """利用可能なプロンプト一覧を返す"""
        result = await self._session.list_prompts()
        return result.prompts

    async def get_prompt(self, prompt_name: str, args: dict[str, str]) -> list:
        """引数を補間したプロンプトのメッセージリストを返す"""
        result = await self._session.get_prompt(prompt_name, args)
        return result.messages

    async def list_tools(self) -> list:
        """ツール一覧を返す（プロンプトとの組み合わせ用）"""
        result = await self._session.list_tools()
        return result.tools

    async def call_tool(self, name: str, args: dict):
        """ツールを呼び出して結果テキストを返す"""
        result = await self._session.call_tool(name, args)
        return result.content[0].text

print("PromptMCPClient 定義完了")

PromptMCPClient 定義完了


In [182]:
import sys
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

async def simulate_cli(prompt_name: str, args: dict[str, str]):
    """
    CLI風の操作をシミュレートする。
    ユーザーが /prompt_name を入力して引数を渡すイメージ。
    """
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["mcp_server.py"],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            mcp_client = PromptMCPClient(session)

            # / を入力したときのオートコンプリート表示
            print("=== '/' 入力時：利用可能なプロンプト一覧 ===")
            prompts = await mcp_client.list_prompts()
            for p in prompts:
                arg_names = [a.name for a in (p.arguments or [])]
                print(f"  /{p.name} {arg_names} - {p.description}")

            # プロンプトを選択して引数を渡す
            print(f"\n=== '/{prompt_name} {args}' を実行 ===")
            messages = await mcp_client.get_prompt(prompt_name, args)

            # メッセージをClaude用の形式に変換
            claude_messages = [
                {"role": m.role, "content": m.content.text}
                for m in messages
            ]

            # ツール一覧も取得（プロンプトがツールを必要とする場合のため）
            tools = await mcp_client.list_tools()
            tools_for_claude = [
                {"name": t.name, "description": t.description, "input_schema": t.inputSchema}
                for t in tools
            ]

            # Claudeに送信
            response = client.messages.create(
                model=model,
                max_tokens=512,
                tools=tools_for_claude,
                messages=claude_messages,
            )

            # ツール呼び出しが発生した場合は処理する
            while response.stop_reason == "tool_use":
                tool_uses = [b for b in response.content if b.type == "tool_use"]
                tool_results = []
                for tool_use in tool_uses:
                    print(f"  → ツール呼び出し: {tool_use.name}({tool_use.input})")
                    result_text = await mcp_client.call_tool(tool_use.name, tool_use.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": tool_use.id,
                        "content": result_text,
                    })
                claude_messages += [
                    {"role": "assistant", "content": response.content},
                    {"role": "user", "content": tool_results},
                ]
                response = client.messages.create(
                    model=model, max_tokens=512,
                    tools=tools_for_claude, messages=claude_messages,
                )

            print(f"\nClaudeの回答:\n{response.content[0].text}")


# /summarize doc_id=api-guide を実行
await simulate_cli("summarize", {"doc_id": "api-guide"})

=== '/' 入力時：利用可能なプロンプト一覧 ===
  /summarize ['doc_id'] - ドキュメントの内容を簡潔に要約する。
  /weather-report ['city'] - 指定した都市の天気を丁寧なレポート形式で回答する。

=== '/summarize {'doc_id': 'api-guide'}' を実行 ===

Claudeの回答:
ドキュメントを要約します：

- APIガイドのドキュメント
- 認証方式：Bearerトークンを使用
- APIの使い方を説明する目的
